# _____NHS Scotland Prescription Analysis – August 2025______


In [1]:
# Step 1:  Install or Upgrade the Folium Library for geospatial mapping.

!pip install folium --upgrade  


In [2]:
# Step 2:  Installing Folium Using the Correct Jupyter Python Environment

import sys
!{sys.executable} -m pip install folium


In [3]:
# Step 3: Import required Libraries
import pandas as pd   # For data handling
import folium         # For interactive maps



In [4]:
import urllib.request
import json
import pandas as pd

# -----------------------------
# Step 4 (API version): Load data from NHS Open Data API instead of CSV
# -----------------------------

API_URL = (
    "https://www.opendata.nhs.scot/api/3/action/datastore_search"
    "?resource_id=381166dd-3a07-4c12-93c3-6db7b12c042a"
)

def load_prescriptions_from_api(max_rows: int | None = None) -> pd.DataFrame:
    """
    Load prescription data directly from the NHS Open Data API.

    - Uses pagination with offset/limit.
    - If max_rows is None → loads all available rows.
    - If max_rows is a number (e.g. 50000) → stops after that many.
    """
    all_records = []
    limit = 10000      # how many rows per API call
    offset = 0

    while True:
        # Stop if we’ve reached the requested max_rows
        if max_rows is not None and len(all_records) >= max_rows:
            break

        url = f"{API_URL}&limit={limit}&offset={offset}"

        # Call the API
        with urllib.request.urlopen(url) as response:
            raw_data = response.read()

        # JSON → Python dict
        data = json.loads(raw_data.decode("utf-8"))

        # Extract actual rows
        records = data["result"]["records"]

        # If there are no more rows, stop
        if not records:
            break

        all_records.extend(records)
        offset += limit

        # Safety: trim to max_rows if set
        if max_rows is not None and len(all_records) >= max_rows:
            all_records = all_records[:max_rows]
            break

    # List[dict] → DataFrame
    df = pd.DataFrame(all_records)
    return df


# 🔹 Use this instead of pd.read_csv(...)
#    Adjust max_rows if needed (None = full dataset, might be large)
df_prescriber_loc = load_prescriptions_from_api(max_rows=50000)

print("Data loaded from NHS API!")
print("Shape:", df_prescriber_loc.shape)
print(df_prescriber_loc.head())


Data loaded from NHS API!
Shape: (50000, 11)
     _id        HBT  GPPractice       DMDCode      BNFItemCode  \
0  28026  S08000015       80306  3.282411e+15  0601011A0BBADAC   
1  28027  S08000015       80306  3.284511e+15  0601012V0BBAAAA   
2  28028  S08000015       80306  3.284811e+15  0601011L0BBACAC   
3  28029  S08000015       80306  3.289811e+15  0603020G0AAADAD   
4  28030  S08000015       80306  3.293891e+16  0403030E0AAANAN   

                                  BNFItemDescription PrescribedType  \
0  NOVORAPID FLEXPEN 100UNITS/ML INJ 3ML PRE-FILL...            AMP   
1  LANTUS 100UNITS/ML SOLUTION FOR INJECTION 3ML ...            AMP   
2  HUMALOG 100UNITS/ML SOLUTION FOR INJECTION 3ML...            AMP   
3                          DEXAMETHASONE 2MG TABLETS            VMP   
4                           FLUOXETINE 40MG CAPSULES            VMP   

   NumberOfPaidItems  PaidQuantity  GrossIngredientCost  PaidDateMonth  
0                 25         155.0               948.60   

# ****** Exploratory Data Analysis ******

In [5]:

# Step 5: Preview the First Five Rows of the Dataset
df_prescriber_loc.head()

,_id,HBT,GPPractice,DMDCode,BNFItemCode,BNFItemDescription,PrescribedType,NumberOfPaidItems,PaidQuantity,GrossIngredientCost,PaidDateMonth
0,28026,S08000015,80306,3.282411e+15,0601011A0BBADAC,NOVORAPID FLEXPEN 100UNITS/ML INJ 3ML PRE-FILL...,AMP,25,155.0,948.60,202508
1,28027,S08000015,80306,3.284511e+15,0601012V0BBAAAA,LANTUS 100UNITS/ML SOLUTION FOR INJECTION 3ML ...,AMP,2,15.0,104.25,202508
2,28028,S08000015,80306,3.284811e+15,0601011L0BBACAC,HUMALOG 100UNITS/ML SOLUTION FOR INJECTION 3ML...,AMP,6,85.0,481.27,202508
3,28029,S08000015,80306,3.289811e+15,0603020G0AAADAD,DEXAMETHASONE 2MG TABLETS,VMP,2,100.0,8.46,202508
4,28030,S08000015,80306,3.293891e+16,0403030E0AAANAN,FLUOXETINE 40MG CAPSULES,VMP,2,90.0,5.79,202508


In [6]:
# Step 3: See all column names
df_prescriber_loc.columns

Index(['_id', 'HBT', 'GPPractice', 'DMDCode', 'BNFItemCode',
       'BNFItemDescription', 'PrescribedType', 'NumberOfPaidItems',
       'PaidQuantity', 'GrossIngredientCost', 'PaidDateMonth'],
      dtype='object')

*** Note:


⭐ Understanding Your Dataset

1️⃣ HBT — Health Board Territory
This tells which NHS Health Board the prescription belongs to.

2️⃣ GPPractice — GP Practice Code
Every GP practice has a unique code.
Example:
80005

3️⃣ DMDCode — Medicine Code
This is a code for the medicine in the NHS Dictionary of Medicines & Devices (DM+D).
Example:
1001411e+15

4️⃣ BNFItemCode — Drug Code (BNF = British National Formulary)
Another code used to identify the specific medicine.
Example:
1001010P0AAAHAH → Naproxen
1310012P0AAAABAB → Fusidic Acid Cream
👉 The BNFItemDescription column tells the actual name.

5️⃣ BNFItemDescription — Medicine Name
This is the real human-readable name of the medicine.
Examples:
“NAPROXEN 250MG GASTRO-RESISTANT TABLETS”

6️⃣ PrescribedType — VMP or AMP
VMP = Virtual Medicinal Product
(generic or grouped medicine)

AMP = Actual Medicinal Product
(specific brand)
Example:
Paracetamol 500mg (generic) → VMP
“Panadol 500mg tablets” (brand) → AMP

7️⃣ NumberOfPaidItems
This is the number of times the medicine was prescribed.
Example:
1 → prescribed once
77 → 77 prescriptions
1837 → 1837 prescriptions of same item

8️⃣ PaidQuantity
How many units were dispensed.
Example:
30 tablets
100ml

9️⃣ GrossIngredientCost
💰 The cost to NHS of this medicine item.
Example:
62.06 = £62.06

🔟 PaidDateMonth — Year & Month
The month of the dataset.
Example:
202508 = August 2025


In [7]:
# Step 4: Check the Structre
df_prescriber_loc.shape

(50000, 11)

In [8]:
# Step 5: Check basic info (data types, missing values, memory use)
df_prescriber_loc.info

<bound method DataFrame.info of           _id        HBT  GPPractice       DMDCode      BNFItemCode  \
0       28026  S08000015       80306  3.282411e+15  0601011A0BBADAC   
1       28027  S08000015       80306  3.284511e+15  0601012V0BBAAAA   
2       28028  S08000015       80306  3.284811e+15  0601011L0BBACAC   
3       28029  S08000015       80306  3.289811e+15  0603020G0AAADAD   
4       28030  S08000015       80306  3.293891e+16  0403030E0AAANAN   
...       ...        ...         ...           ...              ...   
49995  177186  S08000019       25169  1.202111e+15  1003020P0AAACAC   
49996  177187  S08000019       25169  1.204111e+15  1001010P0AAADAD   
49997  177188  S08000019       25169  1.205011e+15  0604011Y0AAAAAA   
49998  177189  S08000019       25169  1.205111e+15  0204000H0AAAJAJ   
49999  177190  S08000019       25169  1.205311e+15  1304000V0AACHCH   

                                      BNFItemDescription PrescribedType  \
0      NOVORAPID FLEXPEN 100UNITS/ML INJ

***  Note: 
  
   this output, we understand:_____

✔ The dataset is very large

✔ It contains rich information about prescriptions

✔ Many columns have missing values

✔ Column types need cleaning


In [9]:
# Step 6: Check for missing values
df_prescriber_loc.isna().sum()

_id                     0
HBT                     0
GPPractice              0
DMDCode                53
BNFItemCode             0
BNFItemDescription      0
PrescribedType          0
NumberOfPaidItems       0
PaidQuantity            0
GrossIngredientCost     0
PaidDateMonth           0
dtype: int64

***  Note: BNFItemDescription	3502	Medicine NAME missing — important column

In [10]:
# Step 7: Remove rows where medicine name is missing
df_prescriber_loc_clean=df_prescriber_loc.dropna(subset=['BNFItemDescription'])
df_prescriber_loc_clean.shape

(50000, 11)

In [11]:
# Step 8:  — Rename important columns , it makes dashboard clean
df_prescriber_loc_clean=df_prescriber_loc_clean.rename(columns={
    'BNFItemDescription': 'Medicine',
    'NumberOfPaidItems': 'Items',
    'GrossIngredientCost': 'Cost',
    'PaidQuantity': 'Quantity',
    'GPPractice': 'Practice',
    'HBT': 'HealthBoard'})
df_prescriber_loc_clean.columns

Index(['_id', 'HealthBoard', 'Practice', 'DMDCode', 'BNFItemCode', 'Medicine',
       'PrescribedType', 'Items', 'Quantity', 'Cost', 'PaidDateMonth'],
      dtype='object')

In [12]:
# Step 9: Check descriptive statistics
df_prescriber_loc_clean.describe()


,_id,Practice,DMDCode,Items,Quantity,Cost,PaidDateMonth
count,50000.000000,50000.000000,4.994700e+04,50000.000000,5.000000e+04,50000.000000,50000.0
mean,92593.203320,57526.105480,1.342433e+16,8.467820,8.475379e+02,97.214493,202508.0
std,50009.597249,31158.270599,1.424715e+16,29.954969,9.252367e+03,1385.657575,0.0
min,28026.000000,16013.000000,9.407110e+14,1.000000,0.000000e+00,0.000000,202508.0
25%,50237.750000,25049.750000,1.314911e+15,1.000000,3.000000e+01,9.530000,202508.0
50%,82535.500000,80344.000000,5.330211e+15,2.000000,1.120000e+02,25.860000,202508.0
75%,124841.250000,80594.000000,2.338286e+16,5.000000,3.900000e+02,73.200000,202508.0
max,179264.000000,99999.000000,4.540411e+16,1541.000000,1.842341e+06,261161.040000,202508.0


In [13]:
# Step 10: Make describe() output easy to read
pd.set_option('display.float_format', '{:,.2f}'.format)
df_prescriber_loc_clean.describe()


,_id,Practice,DMDCode,Items,Quantity,Cost,PaidDateMonth
count,"50,000.00","50,000.00","49,947.00","50,000.00","50,000.00","50,000.00","50,000.00"
mean,"92,593.20","57,526.11","13,424,327,853,866,200.00",8.47,847.54,97.21,"202,508.00"
std,"50,009.60","31,158.27","14,247,153,243,880,986.00",29.95,"9,252.37","1,385.66",0.00
min,"28,026.00","16,013.00","940,711,000,001,101.00",1.00,0.00,0.00,"202,508.00"
25%,"50,237.75","25,049.75","1,314,911,000,001,109.00",1.00,30.00,9.53,"202,508.00"
50%,"82,535.50","80,344.00","5,330,211,000,001,104.00",2.00,112.00,25.86,"202,508.00"
75%,"124,841.25","80,594.00","23,382,861,000,001,104.00",5.00,390.00,73.20,"202,508.00"
max,"179,264.00","99,999.00","45,404,111,000,001,104.00","1,541.00","1,842,341.00","261,161.04","202,508.00"


*** Note: 

Descriptive Statistics – Short Summary

The summary above shows the typical values in the dataset:

Items: Most prescriptions have 1–5 items, with a median of 2. A few rows show very large values (max 4,354), indicating outliers.

Quantity: Normal range is 40–392 units, median 112. Extremely large maximum values suggest special or bulk supplies.

Cost: Most prescriptions cost £9–£72, median £25. Some medicines are very expensive (up to £891k).

PaidDateMonth: All entries belong to August 2025.

  # *** Basic Stats (KPI EDA)   ****

In [14]:
# Step 11: Total prescriptions:
df_prescriber_loc_clean['Items'].sum()

423391

*** Note: NHS Scotland dispensed 9.55 million items in August 2025.

In [15]:
# Step 12: Total NHS Spend (Total NHS Expenditure on Prescriptions)
df_prescriber_loc_clean['Cost'].sum()

4860724.67

*** Note: NHS spent £108.7 million in August 2025 on prescriptions.

In [16]:
# Step 13: Unique Medicines (Number of Distinct Medicines Prescribed)
df_prescriber_loc_clean['Medicine'].nunique()

5856

*** Note: 13.2k different medicines were prescribed across Scotland.

In [17]:
# Step 14: Unique GP Practices (Number of Distinct GP Practices)
df_prescriber_loc_clean['Practice'].nunique()

50

*** Note: Data contains prescriptions from 1,070 GP practices.

# ******  Building Analytical Table (Deep EDA) ***

In [18]:
# Step 15: Top 10 medicines by number of items

top10_items = (
    df_prescriber_loc_clean.groupby('Medicine')['Items']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


top10_items   
             

Medicine
OMEPRAZOLE 20MG GASTRO-RESISTANT CAPSULES         13467
AMLODIPINE 5MG TABLETS                             6604
PARACETAMOL 500MG CAPLETS                          6516
CO-CODAMOL 30MG/500MG CAPLETS                      6424
ASPIRIN 75MG DISPERSIBLE TABLETS                   5355
ATORVASTATIN 20MG TABLETS                          5219
SALBUTAMOL 100MICROGRAMS/DOSE INHALER CFC FREE     4726
LANSOPRAZOLE 30MG GASTRO-RESISTANT CAPSULES        4191
AMLODIPINE 10MG TABLETS                            3894
CLOPIDOGREL 75MG TABLETS                           3734
Name: Items, dtype: int64

*** Note:
The table above shows the medicines with the highest number of prescription items in August 2025.
A prescription item represents one prescription issued, regardless of the quantity dispensed.
These top medicines indicate the most commonly used treatments across Scotland.

In [19]:
# Step 16:  Top 10 medicines by cost
top10_cost = (
    df_prescriber_loc_clean.groupby('Medicine')['Cost']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top10_cost

Medicine
XTANDI 40MG TABLETS                                           263,895.71
GLECAPREVIR 100MG / PIBRENTASVIR 40MG TABLETS                 142,930.26
FREESTYLE LIBRE 2 PLUS SENSOR                                 118,500.00
FORXIGA 10MG TABLETS                                           83,630.50
TRELEGY ELLIPTA 92MICROG/55MICROG/22MICROG/DOSE DRY PDR INH    54,423.50
TRIMBOW 87MICROG/DOSE / 5MICROG/DOSE / 9MICROG/DOSE INH        47,615.00
ESPRANOR 8MG ORAL LYOPHILISATES                                45,473.26
DAPAGLIFLOZIN 10MG TABLETS                                     37,750.81
BUVIDAL 96MG/0.27ML PROLONGED-RELEASE INJ PF SYRINGES          35,475.60
LIDOCAINE 700MG MEDICATED PLASTERS                             31,902.55
Name: Cost, dtype: float64

*** Note:
'The list above shows the medicines that generated the highest overall spending for NHS Scotland in August 2025.

# *****.  Health Board Analysis (Needed for Choropleth + Bar Charts) ****

In [20]:
# Step 17: Total Number of Prescribed Items per NHS Health Board
items_by_board = (
    df_prescriber_loc_clean.groupby('HealthBoard')['Items']
    .sum()
    .sort_values(ascending=False)
)

items_by_board


HealthBoard
S08000015    282669
S08000017     68972
S08000019     66407
S08000016      5343
Name: Items, dtype: int64

*** Note: The output shows the total number of prescription items issued by each NHS Health Board in Scotland.
A prescription item represents one prescription written, regardless of quantity.

In [21]:
# Step 18: Total NHS Cost per Health Board
cost_by_board = (
    df_prescriber_loc_clean.groupby('HealthBoard')['Cost']
    .sum()
    .sort_values(ascending=False)
)

cost_by_board


HealthBoard
S08000015   3,361,190.86
S08000019     814,053.45
S08000017     623,947.55
S08000016      61,532.81
Name: Cost, dtype: float64

*** Note: This table shows the total cost of all prescriptions issued in each NHS Health Board.
It highlights where NHS Scotland spends the most on medicines.

In [22]:
# Step 19: Find high-cost medicines with low usage
high_cost_low_qty = df_prescriber_loc_clean[
    (df_prescriber_loc_clean['Quantity'] < 5) &
    (df_prescriber_loc_clean['Cost'] > 200)
]

high_cost_low_qty[['Medicine','Quantity','Cost']]\
    .sort_values('Cost', ascending=False)\
    .head(20)


,Medicine,Quantity,Cost
46428,NORDITROPIN FLEXPRO 15MG/1.5ML INJ PRE-FILLED ...,4.00,"1,276.20"
40238,XEPLION 150MG/1.5ML PROLONGED-RELEASE INJ PF S...,3.00,"1,177.77"
44977,SANDOSTATIN LAR 30MG INJ VIALS,1.00,998.41
37285,LANREOTIDE 120MG/0.5ML INJ PRE-FILLED SYRINGES,1.00,937.00
30110,SOMATULINE AUTOGEL 120MG/0.5ML INJ PFS WITH SA...,1.00,937.00
19449,PROSTAP 3 DCS 11.25MG INJ PRE-FILLED SYRINGES,4.00,902.88
2956,PROSTAP 3 DCS 11.25MG INJ PRE-FILLED SYRINGES,4.00,902.88
16007,PROSTAP 3 DCS 11.25MG INJ PRE-FILLED SYRINGES,4.00,902.88
41485,ARANESP SURECLICK 150MICROGRAMS/0.3ML INJ PRE-...,4.00,880.88
7722,NORDITROPIN FLEXPRO 10MG/1.5ML INJ PRE-FILLED ...,4.00,850.80


*** Note: The output highlights medicines that were dispensed in very small quantities (less than 5 units) but still had a very high total cost (over £2,000 per prescription).